# Workshop 1 — ETL (G01)
## Data Profiling y Borrador de Modelado Dimensional

Este notebook cubre:
- **Task 1**: Perfilado inicial de los datos (`data_profiling`).
- **Task 2 (borrador)**: Selección del proceso de negocio, declaración del grano, dimensiones y hechos preliminares.

Este documento es un **borrador de trabajo**. Las decisiones finales se documentan en `README.md`.

Fuente de datos: `data/raw/candidates.csv` (no se modifica; toda transformación se hace sobre una copia en memoria).


## 0. Extract — Carga del archivo fuente
Se carga el CSV original sin aplicar transformaciones de negocio, preservando el archivo fuente en `data/raw/`.

In [7]:
import pandas as pd

RAW_PATH = "../data/raw/candidates.csv"

df_raw = pd.read_csv(RAW_PATH, sep=";")
df = df_raw.copy()

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.head()


Filas: 50000
Columnas: 10


,First Name,Last Name,Email,Application Date,Country,YOE,Seniority,Technology,Code Challenge Score,Technical Interview Score
0,Bernadette,Langworth,leonard91@yahoo.com,2021-02-26,Norway,2,Intern,Data Engineer,3,3
1,Camryn,Reynolds,zelda56@hotmail.com,2021-09-09,Panama,10,Intern,Data Engineer,2,10
2,Larue,Spinka,okey_schultz41@gmail.com,2020-04-14,Belarus,4,Mid-Level,Client Success,10,9
3,Arch,Spinka,elvera_kulas@yahoo.com,2020-10-01,Eritrea,25,Trainee,QA Manual,7,1
4,Larue,Altenwerth,minnie.gislason@gmail.com,2020-05-20,Myanmar,13,Mid-Level,Social Media Community Management,9,7


## 1. Task 1 — Perfilado Inicial de los Datos

### 1.1 Columnas y tipos de datos

In [8]:
df.dtypes


First Name                   object
Last Name                    object
Email                        object
Application Date             object
Country                      object
YOE                           int64
Seniority                    object
Technology                   object
Code Challenge Score          int64
Technical Interview Score     int64
dtype: object

### 1.2 Valores nulos por columna

In [9]:
df.isnull().sum()


First Name                   0
Last Name                    0
Email                        0
Application Date             0
Country                      0
YOE                          0
Seniority                    0
Technology                   0
Code Challenge Score         0
Technical Interview Score    0
dtype: int64

### 1.3 Registros duplicados
Se revisan duplicados exactos de fila y duplicados por `Email` (posible reaplicación del mismo candidato).

In [10]:
dup_full = df.duplicated().sum()
dup_email = df["Email"].duplicated().sum()

print(f"Filas duplicadas (todas las columnas): {dup_full}")
print(f"Emails duplicados: {dup_email}")


Filas duplicadas (todas las columnas): 0
Emails duplicados: 167


### 1.4 Valores únicos en atributos categóricos relevantes

In [11]:
print("Countries:", df["Country"].nunique())
print("Seniority:", df["Seniority"].nunique())
print("Technology:", df["Technology"].nunique())

df["Seniority"].value_counts()


Countries: 244
Seniority: 7
Technology: 24


Seniority
Intern       7255
Mid-Level    7253
Trainee      7183
Junior       7100
Architect    7079
Lead         7071
Senior       7059
Name: count, dtype: int64

In [12]:
df["Technology"].value_counts()


Technology
Game Development                           3818
DevOps                                     3808
Social Media Community Management          2028
System Administration                      2014
Mulesoft                                   1973
Development - Backend                      1965
Development - FullStack                    1961
Adobe Experience Manager                   1954
Data Engineer                              1951
Security                                   1936
Business Intelligence                      1934
Development - CMS Frontend                 1934
Database Administration                    1933
Client Success                             1927
Design                                     1906
QA Manual                                  1902
Technical Writing                          1901
QA Automation                              1892
Sales                                      1890
Development - Frontend                     1887
Development - CMS Backend    

### 1.5 Rango de fechas de aplicación

In [13]:
df["Application Date"] = pd.to_datetime(df["Application Date"])
print("Fecha mínima:", df["Application Date"].min().date())
print("Fecha máxima:", df["Application Date"].max().date())


Fecha mínima: 2018-01-01
Fecha máxima: 2022-07-04


### 1.6 Rango de los scores

In [14]:
print("Code Challenge Score:", df["Code Challenge Score"].min(), "-", df["Code Challenge Score"].max())
print("Technical Interview Score:", df["Technical Interview Score"].min(), "-", df["Technical Interview Score"].max())


Code Challenge Score: 0 - 10
Technical Interview Score: 0 - 10


### 1.7 Estadísticas descriptivas de atributos numéricos

In [15]:
df[["YOE", "Code Challenge Score", "Technical Interview Score"]].describe()


,YOE,Code Challenge Score,Technical Interview Score
count,50000.000000,50000.000000,50000.000000
mean,15.286980,4.996400,5.003880
std,8.830652,3.166896,3.165082
min,0.000000,0.000000,0.000000
25%,8.000000,2.000000,2.000000
50%,15.000000,5.000000,5.000000
75%,23.000000,8.000000,8.000000
max,30.000000,10.000000,10.000000


### 1.8 Hallazgos principales del perfilado

- El dataset contiene **50,000 filas** y **10 columnas**, sin valores nulos en ninguna columna.
- No existen filas duplicadas exactas, pero hay **167 correos duplicados**, lo que sugiere candidatos que aplicaron más de una vez; esto debe documentarse como decisión de preparación (no se eliminan por defecto, ya que cada fila representa una aplicación, no un candidato único).
- `Country` tiene **244 valores únicos** (alta cardinalidad, consistente con datos ficticios/sintéticos con nombres de país variados).
- `Seniority` tiene 7 categorías relativamente balanceadas (~7,000 registros cada una): Intern, Trainee, Junior, Mid-Level, Senior, Lead, Architect.
- `Technology` tiene 24 categorías, con Game Development y DevOps como las más frecuentes.
- `Application Date` cubre el rango **2018-01-01 a 2022-07-04**.
- `YOE` va de 0 a 30 años, con media ~15.3.
- `Code Challenge Score` y `Technical Interview Score` van de 0 a 10, con medias cercanas a 5 (distribución aproximadamente uniforme).
- Aplicando la regla de negocio (`Code Challenge Score >= 7` AND `Technical Interview Score >= 7`), aproximadamente el **13.4% de las aplicaciones (6,698) resultarían HIRED**.


## 2. Task 2 (borrador) — Modelo Dimensional

### Paso 1 — Selección del Proceso de Negocio

**Proceso de negocio**: Evaluación de aplicaciones de candidatos en procesos de reclutamiento técnico.

**Justificación**: El dataset registra eventos discretos y medibles (una aplicación evaluada con dos scores) que determinan un resultado de negocio binario (HIRED / NOT HIRED). Este proceso es el que conecta directamente con R1 (tendencias de contratación), R2 (análisis por tecnología) y R3 (análisis por perfil de candidato).


### Paso 2 — Declaración del Grano (borrador, a validar)

Una fila en la **tabla de hechos (FactApplication)** representa: **una aplicación individual de un candidato a un proceso de reclutamiento, evaluada con un Code Challenge Score y un Technical Interview Score en una fecha determinada.**


### R4 y R5 (definidos)

- **R4 — Experiencia vs. Resultado de Contratación**: analizar si mayor experiencia (YOE), controlando por seniority, se asocia con mejores resultados de contratación.
- **R5 — Análisis Geográfico de Reclutamiento**: identificar los países con mayor volumen de aplicaciones y comparar sus tasas de contratación.


### Paso 3 — Dimensiones finales

| Dimensión | Propósito | Atributos principales | Requisito(s) soportado(s) |
|---|---|---|---|
| DimDate | Analizar tendencias temporales de contratación | date_key, full_date, year, month, month_name, quarter | R1 |
| DimTechnology | Comparar resultados de contratación por perfil tecnológico | technology_key, technology_name | R2 |
| DimSeniority | Analizar resultados por nivel de seniority | seniority_key, seniority_name | R3, R4 |
| DimCountry | Analizar volumen y resultados de contratación por país | country_key, country_name | R5 |

`YOE` se modela como atributo numérico degenerado en `FactApplication` (no como dimensión), ya que es un valor casi continuo (0-30) sin agrupación natural fija en el origen.


### Paso 4 — Hechos y medidas finales

| Medida | Significado | Fuente / Cálculo | Requisito(s) soportado(s) |
|---|---|---|---|
| code_challenge_score | Puntaje de la prueba técnica | Columna fuente `Code Challenge Score` | R1, R2, R3, R4 |
| technical_interview_score | Puntaje de la entrevista técnica | Columna fuente `Technical Interview Score` | R1, R2, R3, R4 |
| yoe | Años de experiencia del candidato | Columna fuente `YOE` | R4 |
| is_hired | Indicador binario de contratación | `(code_challenge_score >= 7) AND (technical_interview_score >= 7)` | R1, R2, R3, R4, R5 |
| application_count | Conteo de aplicaciones (medida aditiva, 1 por fila) | Grano de la tabla de hechos | R1, R2, R3, R4, R5 |


### Paso 6 — Validación contra los requisitos

| Requisito | Dimensión(es) requerida(s) | Medida(s) requerida(s) | Soportado |
|---|---|---|---|
| R1 | DimDate | application_count, is_hired | Sí |
| R2 | DimTechnology | application_count, is_hired | Sí |
| R3 | DimSeniority | application_count, is_hired, yoe | Sí |
| R4 | DimSeniority | yoe, is_hired | Sí |
| R5 | DimCountry | application_count, is_hired | Sí |

Las cinco requisitos quedan soportadas sin dimensiones ni medidas sobrantes.


## 3. Validación del modelo dimensional (ejecución del pipeline)
Se ejecuta `src/extract.py`, `src/transform.py` y `src/dimensional_model.py` como borrador para confirmar que el diseño anterior es implementable sobre los datos reales, antes de documentarlo formalmente en `README.md`.

In [10]:
import sys
sys.path.insert(0, "../src")

from extract import extract
from transform import transform
from dimensional_model import build_dimensional_model

raw = extract()
prepared = transform(raw)
model = build_dimensional_model(prepared)

for name, table in model.items():
    print(f"{name}: {table.shape[0]} filas, {table.shape[1]} columnas")


dim_date: 1646 filas, 7 columnas
dim_technology: 24 filas, 2 columnas
dim_seniority: 7 filas, 2 columnas
dim_country: 244 filas, 2 columnas
fact_application: 50000 filas, 10 columnas


In [11]:
model["fact_application"].head()


,application_id,date_key,technology_key,seniority_key,country_key,candidate_email,yoe,code_challenge_score,technical_interview_score,is_hired
0,1,20210226,5,2,162,leonard91@yahoo.com,2,3,3,False
1,2,20210909,5,2,167,zelda56@hotmail.com,10,2,10,False
2,3,20200414,4,5,20,okey_schultz41@gmail.com,4,10,9,True
3,4,20201001,17,7,66,elvera_kulas@yahoo.com,25,7,1,False
4,5,20200520,22,5,148,minnie.gislason@gmail.com,13,9,7,True


### Próximos pasos

- El diseño se trasladó a `src/dimensional_model.py`, `sql/create_tables.sql` (PostgreSQL) y `sql/analytical_queries.sql` (R1-R5).
- La carga a PostgreSQL se implementa en `src/load.py`, orquestada por `src/main.py`.
- La documentación formal de este borrador se encuentra en `README.md`.
